# Apache Spark com Delta Lake

Demonstração de operações CRUD (INSERT, UPDATE, DELETE) com **PySpark** e **Delta Lake**.

**Cenário:** Sistema de Gestão de Vendas — TechStore  
**Tabelas:** clientes, produtos, pedidos

In [ ]:
import shutil, os

# Limpeza antes de iniciar o Spark
# Evita conflito com execucoes anteriores
for d in ['spark-warehouse', 'metastore_db']:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f'Removido: {d}')
    else:
        print(f'{d} nao encontrado — ok')


In [ ]:
from pyspark.sql import SparkSession
from delta import *
import logging

logging.getLogger('py4j').setLevel(logging.WARNING)


In [ ]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0')
    .config('spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark


## Cenário — TechStore

Sistema de gestão de vendas de uma loja de eletrônicos com três entidades.

### Modelo ER
```
CLIENTES (1) ----< PEDIDOS >---- (N) PRODUTOS
```

- Um cliente pode realizar vários pedidos
- Um produto pode estar em vários pedidos

## DDL — Criação das Tabelas Delta

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS clientes (
        id       INT,
        nome     STRING,
        email    STRING,
        cidade   STRING,
        estado   STRING
    )
    USING delta
""")
spark.sql("SELECT * FROM clientes").show()


In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS produtos (
        id        INT,
        nome      STRING,
        categoria STRING,
        preco     FLOAT,
        estoque   INT
    )
    USING delta
""")
spark.sql("SELECT * FROM produtos").show()


In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS pedidos (
        id           INT,
        cliente_id   INT,
        produto_id   INT,
        quantidade   INT,
        data_pedido  STRING,
        status       STRING
    )
    USING delta
""")
spark.sql("SELECT * FROM pedidos").show()


## INSERT — Inserindo Dados

In [ ]:
spark.sql("""
    INSERT INTO clientes VALUES
        (1, 'Ana Silva',       'ana@email.com',      'Sao Paulo',      'SP'),
        (2, 'Carlos Oliveira', 'carlos@email.com',   'Rio de Janeiro', 'RJ'),
        (3, 'Maria Santos',    'maria@email.com',    'Curitiba',       'PR'),
        (4, 'Joao Costa',      'joao@email.com',     'Porto Alegre',   'RS'),
        (5, 'Fernanda Lima',   'fernanda@email.com', 'Belo Horizonte', 'MG')
""")
spark.sql("SELECT * FROM clientes").show()


In [ ]:
spark.sql("""
    INSERT INTO produtos VALUES
        (1, 'Notebook Dell',      'Informatica',  3599.99, 15),
        (2, 'Smartphone Samsung', 'Celulares',    1299.00, 50),
        (3, 'Monitor LG 27',      'Informatica',   899.90, 30),
        (4, 'Teclado Mecanico',   'Perifericos',   349.90, 100),
        (5, 'Mouse Logitech',     'Perifericos',   159.90, 80)
""")
spark.sql("SELECT * FROM produtos").show()


In [ ]:
spark.sql("""
    INSERT INTO pedidos VALUES
        (1, 1, 2, 2, '2024-01-10', 'entregue'),
        (2, 2, 1, 1, '2024-01-12', 'entregue'),
        (3, 3, 3, 1, '2024-01-15', 'em_transporte'),
        (4, 1, 4, 1, '2024-01-20', 'processando'),
        (5, 4, 5, 3, '2024-01-22', 'cancelado')
""")
spark.sql("SELECT * FROM pedidos").show()


## Consulta com JOIN

In [ ]:
spark.sql("""
    SELECT
        p.id         AS pedido_id,
        c.nome       AS cliente,
        pr.nome      AS produto,
        p.quantidade,
        p.status,
        p.data_pedido
    FROM pedidos p
    JOIN clientes c  ON p.cliente_id = c.id
    JOIN produtos pr ON p.produto_id = pr.id
    ORDER BY p.id
""").show(truncate=False)


## UPDATE — Atualizando Dados

In [ ]:
# Atualiza status do pedido 3 para entregue
spark.sql("UPDATE pedidos SET status = 'entregue' WHERE id = 3")
spark.sql("SELECT * FROM pedidos WHERE id = 3").show()


In [ ]:
# Ajusta preco e estoque do produto 2
spark.sql("UPDATE produtos SET preco = 1199.00, estoque = 45 WHERE id = 2")
spark.sql("SELECT * FROM produtos WHERE id = 2").show()


## DELETE — Removendo Dados

In [ ]:
# Remove pedidos cancelados
spark.sql("DELETE FROM pedidos WHERE status = 'cancelado'")
spark.sql("SELECT * FROM pedidos").show()


## ALTER TABLE — Evolução de Schema

O Delta Lake suporta adicionar colunas sem recriar a tabela.

In [ ]:
spark.sql("ALTER TABLE clientes ADD COLUMN telefone STRING")
spark.sql("SELECT * FROM clientes").show()


In [ ]:
spark.sql("UPDATE clientes SET telefone = '(11) 91234-5678' WHERE id = 1")
spark.sql("UPDATE clientes SET telefone = '(21) 99876-5432' WHERE id = 2")
spark.sql("SELECT * FROM clientes").show()


## MERGE — Upsert (Insert or Update)

Insere o registro se não existir, ou atualiza se já existir.

In [ ]:
spark.sql("""
    MERGE INTO clientes AS target
    USING (
        SELECT 6 AS id, 'Pedro Alves' AS nome, 'pedro@email.com' AS email,
               'Fortaleza' AS cidade, 'CE' AS estado, '(85) 98765-4321' AS telefone
    ) AS source
    ON target.id = source.id
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")
spark.sql("SELECT * FROM clientes").show()


## Time Travel — Viagem no Tempo

O Delta Lake mantém um **transaction log** que permite consultar versões anteriores.

In [ ]:
# Historico completo de transacoes da tabela clientes
spark.sql("DESCRIBE HISTORY clientes").show(truncate=False)


In [ ]:
from delta.tables import DeltaTable

# Verifica se e tabela Delta
print(DeltaTable.isDeltaTable(spark, 'spark-warehouse/clientes'))

# Le versao 0 (estado inicial — apenas INSERT)
df_v0 = spark.read.format('delta').option('versionAsOf', 0).load('spark-warehouse/clientes')
print('Clientes na versao 0 (so INSERT):')
df_v0.show()


In [ ]:
print('Clientes na versao atual:')
spark.sql("SELECT * FROM clientes").show()


In [ ]:
spark.stop()
print('Sessao Spark encerrada.')
